# Module 5: Giving the agent knowledge it can search

The agent answers a maintenance question from its own guesses. You give it a search tool over Kestrel's service bulletins, check that its answer follows the bulletin it found, and see what it does when the bulletins don't cover a question.

In [ ]:
import labnb as lab
%cd ../agent

## Start from module 5

This puts the agent in the state module 5 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
!python reset.py 5

## 1. An answer the model can't know

Fault F-21 is documented only in Kestrel's service bulletins, which no model has seen.

In [ ]:
!python agent.py --q m5
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))
lab.trace_link()

A specific, plausible answer. Module 4's contract doesn't catch it, because no tool failed. In the
trace, look for a retrieval step: there isn't one, because the agent has no way to search yet.

## 2. The knowledge it could use

The bulletins are already loaded into a PostgreSQL database with pgvector in your project. pgvector is
a PostgreSQL extension that stores each passage as a vector, a list of numbers, so the database can
rank passages by how closely they match a question. Here the vectors come from the passages' words
(see [`retrieval.py`](../agent/retrieval.py)); a real system would use an embedding model. Query it
directly, outside the agent:

In [ ]:
!python retrieval.py "fault F-21"

Each passage carries its `source`, the bulletin it came from. That's what makes an answer
checkable.

## 3. Register the search tool

In [ ]:
!cp solutions/module-05/tools.py solutions/module-05/agent.py .
!git diff --stat -- tools.py agent.py

`search_bulletins` is declared like the other tools and returns passages with their sources using module 4's contract. The supplied agent instructs the model to cite those source IDs and to acknowledge a search miss instead of inventing an answer.

```mermaid
flowchart LR
  Q["The F-21 question"] --> M["Model"]
  M -- "search_bulletins: fault F-21" --> T["Retrieval tool"]
  T --> DB[("pgvector: Kestrel service bulletins")]
  DB --> T
  T -- "passages with their source ids, in module 4's contract" --> M
  M --> ANS["An answer that matches a passage"]
```

## 4. Ask again

In [ ]:
!python agent.py --q m5
lab.tool_results()
lab.trace_link()

**This is the module's success signal:** `search_bulletins` was called, passages came back from
SB-2024-102, and the answer follows the passage ("coolant loop pressure low") rather than the
model's guess.

## 5. Ask something the bulletins don't cover

There is no Kestrel K-900.

In [ ]:
!python agent.py --q m5-miss
lab.tool_results()
lab.evidence(columns=("question", "tools run", "answer"), last=3)

The search returns `no_results`, and the agent says so instead of inventing. Compare it with the
first run of this module, which invented.

Grounding makes an answer checkable; it doesn't make it correct. Nothing yet checks that the
retrieved passage actually answers the question. Module 7 comes back to that.

## If the result differs

If the direct retrieval command cannot reach PostgreSQL, stop and check environment readiness; do not substitute a guessed answer. If F-21 returns no passage, confirm that your project has the bulletin corpus and the expected database settings before continuing. After the fix, require a retrieved passage from `SB-2024-102`, its source ID in the answer, and the documented “coolant loop pressure low” cause. Retrieval alone does not prove the answer used it. For K-900, require `search_bulletins` to return `no_results` and the answer to acknowledge the gap without supplying an invented interval.